# Consolidation + Per-Class LoRA Compression for Qwen1.5-0.5B

**Research prototype.** Frozen Qwen1.5-0.5B teacher → consolidated student (shared per-class backbone + per-layer rank-r LoRA) distilled to match it.

> ⚠️ **Set the GPU first:** Runtime → Change runtime type → **T4 GPU**, then Run all.

Reading the result: the question is *not* whether the student beats the teacher (it won't). It's whether a consolidated point sits **left of / below** the 4-bit nf4 baseline in `frontier.png` (lower perplexity = better).

In [ ]:
# 1. Clone the repo
!git clone https://github.com/sinha-k-prat/consolidated-qwen.git
%cd consolidated-qwen

In [ ]:
# 2. Install pinned dependencies (Colab-tested versions)
!pip install -q -r requirements.txt

In [ ]:
# 3. Confirm we actually have a GPU (expect a Tesla T4). If this errors or shows
#    no GPU, go to Runtime > Change runtime type > T4 GPU and re-run.
!nvidia-smi

In [ ]:
# 4. (Optional but recommended) Smoke-test the whole pipeline in ~1 min:
#    10 steps/rank on tiny data, catches shape bugs before the real run.
!python run_sweep.py --smoke-test --ranks 8

In [ ]:
# 5. Real proof-of-concept sweep over ranks [4, 8, 16, 32].
#    Small step counts keep this under ~2h on a T4. --fp16 saves memory.
#    Tweak --steps / --ranks to trade runtime for quality.
!python run_sweep.py --fp16 --ranks 4 8 16 32 --steps 300

In [ ]:
# 6. Show the frontier inline (perplexity vs storage size).
from IPython.display import Image, display
import json
with open('results.json') as f:
    print(json.dumps(json.load(f), indent=2))
display(Image('frontier.png'))